In [ ]:
!pip install mlflow boto3 awscli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [ ]:
# AWS Access Key ID
!aws configure

AWS Access Key ID [None]: a
AWS Secret Access Key [None]: 
Default region name [None]: e
Default output format [None]: r


In [ ]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")

In [ ]:
# create an experiment
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

MlflowException: API request to http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPConnectionPool(host='ec2-52-204-122-132.compute-1.amazonaws.com', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Exp+4+-+Handling+Imbalanced+Data (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x7e0505f85c10>, 'Connection to ec2-52-204-122-132.compute-1.amazonaws.com timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120) to a larger value.

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [ ]:
df=pd.read_csv("reddit_preproccessing.csv").dropna(subset=["clean_comment"])
df.shape

In [ ]:
# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_methode):
  ngram_range=(1,3)
  max_features=10000

  X_train, X_test, y_train,y_test=train_test_split(df["clean_comment"],df["category"],test_size=0.2,random_state=42)
  # Step 2: Vectorization using TF-IDF with max_features
  vectorizer=TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)
  X_train_vec=vectorizer.fit_transform(X_train)
  X_test_vec=vectorizer.transform(X_test)

  # step 3: Handle class imbalance based on the selected method (only applied to the training set)
  if imbalance_methode == "class_weights":
     class_weight= "balanced" # Use class_weight in Random Forest
  else:
     class_weight=None # Do not apply class_weight if using resampling

     # Resampling Techinque (only apply to the training set)
     if imbalance_methode == "oversampling":
        smote=SMOTE(random_state=42)
        X_train_vec, y_train= smote.fit_resample(X_train_vec, y_train)
     elif imbalance_methode == "adasyn":
        adasyn=ADASYN(random_state=42)
        X_train_vec, y_train= adasyn.fit_resample(X_train_vec, y_train)
     elif imbalance_methode == "undersampling":
        rus=RandomUnderSampler(random_state=42)
        X_train_vec, y_train= rus.fit_resample(X_train_vec, y_train)
     elif imbalance_methode == "smote_enn":
        smote_enn =SMOTEENN(random_state=42)
        X_train_vec, y_train= smote_enn.fit_resample(X_train_vec, y_train)

  with mlflow.start_run() as run:
      # Set tags for the experiment and add description
      mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_methode}_RandomForest_TFIDF_Trigrams")
      mlflow.set_tag("experiment_type", "imbalance_handling")
      mlflow.set_tag("model_type", "RandomForestClassifier")
      mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_methode}")

      # Log vectorizer parameters
      mlflow.log_param("vectorizer_type", "TF-IDF")
      mlfow.log_param("ngram_range", ngram_range)
      mlflow.log_param("vectorizer_max_features", max_features)

      # Log Random Forest parameters
      n_estimators=200
      max_depth=15

      mlflow.log_param("n_estimators", n_estimators)
      mlflow.log_param("max_depth", max_depth)
      mlflow.log_param("imbalance_method", imbalance_method)

      # Initialize and train the model
      model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
      model.fit(X_train, y_train)

      # Step 5: Make predictions and log metrics
      y_pred=model.predict(X_test)

      # Log accuracy
      accuracy=accuracy_score(y_test,y_pred)
      mlflow.log_metric("accuracy", accuracy)
      # Log classification report
      classification_rep=classification_report(y_test,y_pred,out_dict=True)
      for label, metrics in classification_rep.items():
         if isinstance(metrics,dict):
           for metric, value in metrics.items():
              mlflow.log_metric(f"{label}_{metric}",value)

      # Log confusion matrix
      conf_matrix=confusion_matrix(y_test,y_pred)
      plt.figure(figsize=(8,6))
      sns.heatmap(conf_matrix,annot=True,fmt="d",cmap='Blues')
      plt.xlabel("Predicted")
      plt.ylabel("Actual")
      plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
      plt.savefig("confusion_matrix.png")
      mlflow.log_artifcat("confusion_matrix.png")
      plt.close()

# Step 7: Run experiments for different imbalance methods
imbalance_methods=["class_weights","oversampling","adasyn","undersampling","smote_enn"]

for method in imbalance_methods:
    run_imbalanced_experiment(method)


